In [76]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "dufour2009calculated")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "transfers.csv")
complete_path_2 = os.path.join(original_data_pathway, "gestures.csv")
complete_path_3 = os.path.join(original_data_pathway, "alternations.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [77]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)

df1['study_id']="dufour2009calculated"
df1.columns = map(str.lower, df1.columns)
df1=df1.applymap(lambda s: s.lower() if type(s) == str else s)
# df1.columns


In [78]:

df1_temp = df1[['study_id','participant', 'session', 'trial', 
        'td pv_b2d', 'td novalue','td bp_b2d', 'ti pv_b2d', 
        'ti novalue_b2d', 'ti bp_b2d', 'tp pv_b2d','tp novalue_b2d', 
        'tpbp_b2d', 'catchpv_b2d', 'catchnov_b2d','catchbp_b2d', 
        'comments','pagai to bimbo', 'bimbo to pagai']].values.tolist() + df1[['study_id','participant_2', 'session', 'trial',
       'td pv_d2b', 'td novalue_d2b','td bp_d2b', 'ti pv_d2b', 
       'ti novalue_d2b', 'ti bp_d2b', 'tp pv_d2b','tp novalue_d2b', 
       'tpbp_d2b', 'catchpv_d2b', 'catchnov_d2b','catchbp_d2b', 
       'comments' ,'pagai to bimbo', 'bimbo to pagai'
             ]].values.tolist()

df1_long = pd.DataFrame(df1_temp, columns=['study_id','participant', 'session', 'trial', 
        'transfer_direct_partner_value', 'transfer_direct_no_value_token', 'transfer_direct_banana_peel','transfer_indirect_partner_value',
        'transfer_indirect_no_value_token','transfer_indirect_banana_peel','transfer_passive_partner_value','transfer_passive_no_value_token',
        'transfer_passive_banana_peel','catch_partner_value_token','catch_partner_no_value_token', 'catch_banana_peel',
        'comments','pagai_to_bimbo', 'bimbo_to_pagai'])

In [79]:
df2 = pd.read_csv(complete_path_2)

df2['study_id']="dufour2009calculated"
df2.columns = map(str.lower, df2.columns)
df2=df2.applymap(lambda s: s.lower() if type(s) == str else s)

df2_temp = df2[['study_id','participant', 'session', 'trial', 
        'nbpointing/xvaluable token','po/nov', 
        'po/nothing foll by 2 items', 'hand/xv',
       'hd/nov_incbananapeel', 'hd/nothing', 
       '0/ xv', '0/nov']].values.tolist() + df2[['study_id','participant.1', 'session', 'trial',
        'nbpo/xv', 'po/nov.1', 
        'po/nothing', 'hd/xv', 
        'hd/nov','hd/nothing.1', 
        '0/ xv.1', '0/nov.1',
             ]].values.tolist()

df2_long = pd.DataFrame(df2_temp, columns=['study_id','participant_test', 'session_test', 'trial_test', 
        'nb_pointing_x_valuable_token', 'po_nov',
        'po_nothing_followed_by_2_items','hand_x_v',
        'hd_nov_inc_banana_peel','hd_nothing',
        '0_xv','nov'])
# df2.columns

In [80]:
df1_long_list = df1_long.values.tolist()
df2_long_list = df2_long.values.tolist()

# print(len(df1_long_list))
# print(len(df2_long_list))

combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(df1_long_list,df2_long_list)] ##list comprehension is your friend
# print(combined_lol) 
fulldf = pd.DataFrame(combined_lol, columns=['study_id','participant_1', 'session', 'trial', 
        'transfer_direct_partner_value', 'transfer_direct_no_value_token', 'transfer_direct_banana_peel','transfer_indirect_partner_value',
        'transfer_indirect_no_value_token','transfer_indirect_banana_peel','transfer_passive_partner_value','transfer_passive_no_value_token',
        'transfer_passive_banana_peel','catch_partner_value_token','catch_partner_no_value_token', 'catch_banana_peel',
        'comments','pagai_to_bimbo', 'bimbo_to_pagai',
        'study_id_test','participant_test', 'session_test', 'trial_test', 
        'nb_pointing_x_valuable_token', 'po_nov',
        'po_nothing_followed_by_2_items','hand_x_v',
        'hd_nov_inc_banana_peel','hd_nothing',
        '0_xv','nov'])

In [81]:
fulldf.loc[fulldf.participant_1 == 'bimbo', ['participant_2']] = 'dokana'
fulldf.loc[fulldf.participant_1 == 'dokana', ['participant_2']] = 'bimbo'


fulldf['dyad']=fulldf.participant_1.str.cat(fulldf.participant_2, sep='_')

fulldf['present']="pagai"
fulldf['role']="giver"
fulldf['role_2']="receiver"

In [82]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant_1'] = fulldf['participant_1'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant_1'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant_1', right_on='name', how='left')

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='participant_2', right_on='name_2', how='left')




In [83]:
fulldf['session']=fulldf['session'].astype(str)
fulldf = fulldf[~fulldf.session.str.contains('0')]


In [84]:
fulldf.rename(columns={"sex": "sex",
                    "participant_1":'participant',
                    'nb_pointing_x_valuable_token':'pointing_followed_by_valuable_token_transfer', 
                    'po_nov':'pointing_followed_by_nonvaluable_token_transfer',
                    'hand_x_v':'hand_beg_followed_by_valuable_token', 
                    'po_nothing_followed_by_2_items':"pointing_followed_by_nothing",
                    'hd_nov_inc_banana_peel':'hand_beg_followed_by_nonvaluable_token',
                    'hd_nothing':'hand_beg_followed_by_nothing', 
                    '0_xv':'no_gesture_followed_by_valuable_token', 
                    'nov':'no_gesture_followed_by_nonvaluable_token'}, inplace=True)

# fulldf.columns

replace_list=['pointing_followed_by_valuable_token_transfer','pointing_followed_by_nonvaluable_token_transfer',
     'hand_beg_followed_by_valuable_token']

# fulldf['pointing_followed_by_valuable_token_transfer'].unique()

for x in replace_list:
    fulldf[x].replace('\/', 'x_1gesture_and_tokens', inplace=True, regex=True)
    fulldf[x].replace('\'', '', inplace=True, regex=True)
    fulldf[x].replace('\,', 'x_1gesture_1token_and_', inplace=True, regex=True)

for x in replace_list:
     fulldf[x].replace('tokens2', '2tokens', inplace=True, regex=True)
     fulldf[x].replace('tokens4', '4tokens', inplace=True, regex=True)
     fulldf[x].replace('tokens3', '3tokens', inplace=True, regex=True)

In [85]:
replace_list_2=[['3tdpv','3x_transfer_direct_partner_value'],
                ['1tinov','1x_transfer_indirect_no_value'],
                ['2tdpv&4tipv','2x_transfer_direct_partner_value_and_4x_transfer_indirect_partner_value'],
                ['1tdpv & 3tppv','1x_transfer_direct_partner_value_and_3x_transfer_passive_partner_value'],
                [ '1tipv','1x_transfer_indirect_partner_value'],
                ['5tdpv','5x_transfer_direct_partner_value'],
                ['1td nov','1x_transfer_direct_no_value'],
                ['2tdpv','2x_transfer_direct_partner_value'],
                ['3tdbp','3x_transfer_direct_banana_peel'],
                ['1tdbp ','1x_transfer_direct_banana_peel']]
for x,y in replace_list_2:
     fulldf['pagai_to_bimbo'].replace(x, y, inplace=True, regex=True)
     fulldf['bimbo_to_pagai'].replace(x, y, inplace=True, regex=True)
# fulldf['bimbo_to_pagai'].unique()



In [86]:
df3 = pd.read_csv(complete_path_3)

df3.columns = map(str.lower, df3.columns)
df3=df3.applymap(lambda s: s.lower() if type(s) == str else s)


output=[]
df3_transform = [['bimbo', 'serie 1_b_gen','serie 1_b_val', 
                                    'serie 2_b_gen','serie 2_b_val',
                                    'serie 3_b_gen','serie 3_b_val',
                                    'serie 4_b_gen', 'serie 4_b_val'],
                ['dokana', 'serie 1_d_gen','serie 1_d_val', 
                                    'serie 2_d_gen','serie 2_d_val',
                                    'serie 3_d_gen','serie 3_d_val',
                                    'serie 4_d_gen', 'serie 4_d_val']]
for name, a,b,c,d,e,f,g,h in df3_transform:
    series1 = df3[['test_num', a, b]].values.tolist() 
    for entry in series1:
        entry+=[name, '1'] 
    series2 = df3[['test_num', c, d]].values.tolist() 
    for entry in series2:
        entry+=[name, '2'] 
    series3 = df3[['test_num', e, f]].values.tolist() 
    for entry in series3:
        entry+=[name, '3'] 
    series4 = df3[['test_num', g, h]].values.tolist() 
    for entry in series4:
        entry+=[name, '4']
    series1 += series2 
    series3 += series4
    series1 += series3
    output += series1 #
df3_new = pd.DataFrame(output, columns=['test_num', 'alternations_general', 'alternations_valuable', 'participant', 'session_number'])
# df3_new.columns


In [87]:
fulldf=fulldf[['study_id', 'participant', 'sex','role', 
    'participant_2','sex_2','role_2', 'present','species', 'dyad', 'session',  'trial',
       'transfer_direct_partner_value', 'transfer_direct_no_value_token',
       'transfer_direct_banana_peel', 'transfer_indirect_partner_value',
       'transfer_indirect_no_value_token', 'transfer_indirect_banana_peel',
       'transfer_passive_partner_value', 'transfer_passive_no_value_token',
       'transfer_passive_banana_peel', 'catch_partner_value_token',
       'catch_partner_no_value_token', 'catch_banana_peel',  'pointing_followed_by_valuable_token_transfer',
        'pointing_followed_by_nonvaluable_token_transfer', 'pointing_followed_by_nothing',
       'hand_beg_followed_by_valuable_token', 'hand_beg_followed_by_nonvaluable_token',
       'hand_beg_followed_by_nothing', 'no_gesture_followed_by_valuable_token', 
       'no_gesture_followed_by_nonvaluable_token','pagai_to_bimbo', 'bimbo_to_pagai']]

fulldf_list =fulldf.values.tolist() 
df3_new_list = df3_new.values.tolist() 
# print(len(fulldf_list))
# print(len(output))  
combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(fulldf_list,df3_new_list)] 
fulldf_new = pd.DataFrame(combined_lol, columns=['study_id', 'participant', 'sex','role', 
    'participant_2','sex_2','role_2', 'present','species', 'dyad', 'session', 'trial',
       'transfer_direct_partner_value', 'transfer_direct_no_value_token',
       'transfer_direct_banana_peel', 'transfer_indirect_partner_value',
       'transfer_indirect_no_value_token', 'transfer_indirect_banana_peel',
       'transfer_passive_partner_value', 'transfer_passive_no_value_token',
       'transfer_passive_banana_peel', 'catch_partner_value_token',
       'catch_partner_no_value_token', 'catch_banana_peel',  'pointing_followed_by_valuable_token_transfer',
        'pointing_followed_by_nonvaluable_token_transfer', 'pointing_followed_by_nothing',
       'hand_beg_followed_by_valuable_token', 'hand_beg_followed_by_nonvaluable_token',
       'hand_beg_followed_by_nothing', 'no_gesture_followed_by_valuable_token', 
       'no_gesture_followed_by_nonvaluable_token','pagai_to_bimbo', 'bimbo_to_pagai',
       'test_num', 'alternations_general', 'alternations_valuable', 'participant_name', 'session_number'])


In [88]:
dufour2009calculated_standardized=fulldf_new[['study_id', 'participant', 'sex','role', 
    'participant_2','sex_2','role_2', 'present','species', 'dyad', 'session',  'trial',
       'transfer_direct_partner_value', 'transfer_direct_no_value_token',
       'transfer_direct_banana_peel', 'transfer_indirect_partner_value',
       'transfer_indirect_no_value_token', 'transfer_indirect_banana_peel',
       'transfer_passive_partner_value', 'transfer_passive_no_value_token',
       'transfer_passive_banana_peel', 'catch_partner_value_token',
       'catch_partner_no_value_token', 'catch_banana_peel',  'pointing_followed_by_valuable_token_transfer',
        'pointing_followed_by_nonvaluable_token_transfer', 'pointing_followed_by_nothing',
       'hand_beg_followed_by_valuable_token', 'hand_beg_followed_by_nonvaluable_token',
       'hand_beg_followed_by_nothing', 'no_gesture_followed_by_valuable_token', 
       'no_gesture_followed_by_nonvaluable_token',
       'alternations_general', 'alternations_valuable','pagai_to_bimbo', 'bimbo_to_pagai']]

comp_out_path_stand = os.path.join(out_pathway, 'dufour2009calculated_standardized.csv')
dufour2009calculated_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =dufour2009calculated_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
dufour2009calculated_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'dufour2009calculated_glossary.csv')
dufour2009calculated_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)